This notebook explores what is the most efficient way to extract the genome assembly version linked to specific transcript accessions that have been removed from NCBI.

In [ ]:
from Bio import Entrez
Entrez.email = "daniel.garciaruano@ibgc.cnrs.fr"

# Step 1: Get the internal UID for the nucleotide accession
handle = Entrez.esearch(db="nuccore", term="XM_001689407.1")
record = Entrez.read(handle)
nuccore_uid = record["IdList"][0]

# Step 2: Link to assembly database
handle = Entrez.elink(dbfrom="nuccore", db="assembly", id=nuccore_uid)
link_record = Entrez.read(handle)
assembly_uid = link_record[0]["LinkSetDb"][0]["Link"][0]["Id"]

# Step 3: Fetch assembly metadata
handle = Entrez.esummary(db="assembly", id=assembly_uid)
summary = Entrez.read(handle, validate=False)
doc = summary["DocumentSummarySet"]["DocumentSummary"][0]
print(doc["AssemblyAccession"])   # e.g., GCF_000002985.6
print(doc["AssemblyName"])        # e.g., WBcel235
print(doc["AnnotReleaseDate"])
print(doc["AnnotationName"])      # e.g., NCBI annotation release 109

IndexError: list index out of range

In [ ]:
nuccore_uid

'159462457'

In [ ]:
link_record

[{'LinkSetDb': [], 'LinkSetDbHistory': [], 'ERROR': [], 'DbFrom': 'nuccore', 'IdList': ['159462457']}]

```
# From BioProject to Assembly
esearch -db bioproject -query "PRJNA12345" | elink -target assembly | efetch -format docsum

# From BioSample to Assembly
conda activate ncbi & esearch -db biosample -query "SAMN02953692" | elink -target assembly | efetch -format docsum
```

In [ ]:
!conda init && conda activate ncbi && esearch -db biosample -query "SAMN02953692" | elink -target assembly | efetch -format docsum

no change     /home/dgarcia/miniforge3/condabin/conda
no change     /home/dgarcia/miniforge3/bin/conda
no change     /home/dgarcia/miniforge3/bin/conda-env
no change     /home/dgarcia/miniforge3/bin/activate
no change     /home/dgarcia/miniforge3/bin/deactivate
no change     /home/dgarcia/miniforge3/etc/profile.d/conda.sh
no change     /home/dgarcia/miniforge3/etc/fish/conf.d/conda.fish
no change     /home/dgarcia/miniforge3/shell/condabin/Conda.psm1
no change     /home/dgarcia/miniforge3/shell/condabin/conda-hook.ps1
no change     /home/dgarcia/miniforge3/lib/python3.12/site-packages/xontrib/conda.xsh
no change     /home/dgarcia/miniforge3/etc/profile.d/conda.csh
no change     /home/dgarcia/.bashrc
No action taken.

CondaError: Run 'conda init' before 'conda activate'



In [ ]:
efetch -db nuccore -id "XM_001689407.1" -format gb | grep -E "DBSOURCE|assembly"

In [ ]:
(Chlamidomonas reinhardtii[Organism]) AND ("2000/01/01"[Publication Date] : "2025/01/01"[Publication Date])

In [ ]:
esearch -db assembly -query "Chlamydomonas reinhardtii[Organism] AND 1990/01/01:2018/01/01[SeqReleaseDate]" | esummary | xtract -pattern DocumentSummary -element AssemblyAccession,AssemblyName,SeqReleaseDate

In [ ]:
import xml.etree.ElementTree as ET
def parse_gene_xml(xml_text):
    """
    Parse a single Entrezgene XML block and extract:
      - gene_status  : 'live' | 'discontinued' | 'unknown'
      - locus_tag    : e.g. CHLREDRAFT_162526
      - scaffold_acc : genomic accession from Entrezgene_locus (e.g. NW_001843471)
      - mrna_accs    : list of XM_/NM_/XR_/NR_ accessions from Entrezgene_locus products
      - protein_accs : list of XP_/NP_ accessions
      - update_date  : last update date string

    XML path for mRNA accession (based on the actual record structure):
      Entrezgene
        └─ Entrezgene_locus
             └─ Gene-commentary  [type genomic]
                  └─ Gene-commentary_products
                       └─ Gene-commentary  [type mRNA, value="3"]
                            └─ Gene-commentary_accession   ← XM accession lives here
    """
    try:
        root = ET.fromstring(xml_text)
    except ET.ParseError:
        return {}

    ns = {}  # no namespace in these records

    result = {
        "gene_status":   "unknown",
        "locus_tag":     "",
        "scaffold_acc":  "",
        "mrna_accs":     [],
        "protein_accs":  [],
        "update_date":   "",
    }

    # --- Gene status ---
    status_el = root.find(".//Gene-track_status")
    if status_el is not None:
        val = status_el.get("value", "")
        result["gene_status"] = val if val else status_el.text or "unknown"

    # --- Locus tag ---
    lt = root.find(".//Gene-ref_locus-tag")
    if lt is not None:
        result["locus_tag"] = lt.text or ""

    # --- Update date ---
    y = root.find(".//Gene-track_update-date//Date-std_year")
    m = root.find(".//Gene-track_update-date//Date-std_month")
    d = root.find(".//Gene-track_update-date//Date-std_day")
    if y is not None:
        result["update_date"] = f"{y.text}/{m.text:>02}/{d.text:>02}"

    # --- Genomic scaffold + mRNA/protein accessions from Entrezgene_locus ---
    # The structure is:
    # Entrezgene_locus
    #   Gene-commentary (type=1, genomic)        ← scaffold
    #     Gene-commentary_accession              ← NW_ / NC_ accession
    #     Gene-commentary_products
    #       Gene-commentary (type=3, mRNA)       ← XM_ accession
    #         Gene-commentary_accession
    #         Gene-commentary_products
    #           Gene-commentary (type=8, peptide) ← XP_ accession
    #             Gene-commentary_accession

    locus_el = root.find("Entrezgene_locus")
    if locus_el is not None:
        for gc_genomic in locus_el.findall("Gene-commentary"):
            gc_type = gc_genomic.find("Gene-commentary_type")
            if gc_type is not None and gc_type.get("value") == "genomic":
                # scaffold accession
                acc_el = gc_genomic.find("Gene-commentary_accession")
                if acc_el is not None and acc_el.text:
                    result["scaffold_acc"] = acc_el.text

                # mRNA products of this genomic commentary
                products_el = gc_genomic.find("Gene-commentary_products")
                if products_el is not None:
                    for gc_mrna in products_el.findall("Gene-commentary"):
                        mrna_type = gc_mrna.find("Gene-commentary_type")
                        if mrna_type is not None and mrna_type.get("value") == "3":
                            mrna_acc = gc_mrna.find("Gene-commentary_accession")
                            mrna_ver = gc_mrna.find("Gene-commentary_version")
                            if mrna_acc is not None and mrna_acc.text:
                                ver = f".{mrna_ver.text}" if mrna_ver is not None else ""
                                result["mrna_accs"].append(f"{mrna_acc.text}{ver}")

                            # protein products of this mRNA
                            prot_products = gc_mrna.find("Gene-commentary_products")
                            if prot_products is not None:
                                for gc_prot in prot_products.findall("Gene-commentary"):
                                    prot_type = gc_prot.find("Gene-commentary_type")
                                    if prot_type is not None and prot_type.get("value") == "8":
                                        prot_acc = gc_prot.find("Gene-commentary_accession")
                                        prot_ver = gc_prot.find("Gene-commentary_version")
                                        if prot_acc is not None and prot_acc.text:
                                            ver = f".{prot_ver.text}" if prot_ver is not None else ""
                                            result["protein_accs"].append(f"{prot_acc.text}{ver}")

    result["mrna_accs"]    = list(dict.fromkeys(result["mrna_accs"]))    # deduplicate
    result["protein_accs"] = list(dict.fromkeys(result["protein_accs"]))
    return result



In [ ]:
gid="5715093"
fetch_handle = Entrez.elink(dbfrom="gene", db="assembly", id=gid)
xml_text = fetch_handle.read()
fetch_handle.close()

In [ ]:
ET.fromstring(xml_text)

# The xml has an elink dtd link. parse it
root = ET.fromstring(xml_text)
ns = {"elink": "http://www.ncbi.nlm.nih.gov/entrez/eutils/elink"}
assembly_uids = [link.find("elink:Id", ns).text for link in root.findall(".//elink:LinkSetDb/elink:Link", ns)]
assembly_uids

b'<?xml version="1.0" encoding="UTF-8" ?>\n<!DOCTYPE eLinkResult PUBLIC "-//NLM//DTD elink 20101123//EN" "https://eutils.ncbi.nlm.nih.gov/eutils/dtd/20101123/elink.dtd">\n<eLinkResult>\n\n  <LinkSet>\n    <DbFrom>gene</DbFrom>\n    <IdList>\n      <Id>5715093</Id>\n    </IdList>\n  </LinkSet>\n</eLinkResult>\n'


In [ ]:
parse_gene_xml(xml_text)

{'gene_status': 'unknown',
 'locus_tag': '',
 'scaffold_acc': '',
 'mrna_accs': [],
 'protein_accs': [],
 'update_date': ''}

In [ ]:
parse_gene_xml